# I-130 Mandamus Audit — Mobile Runner

This notebook pulls the newest code from the private GitHub repo and runs the audit.

Required Colab Secrets: `GITHUB_TOKEN` (fine-grained, read-only access to this repo) and `COURTLISTENER_API_KEY`.

In [ ]:
from google.colab import userdata
import base64, os, pathlib, subprocess, sys

REPO = 'mongkokman91/i130-mandamus-audit'
ROOT = pathlib.Path('/content/i130-mandamus-audit')
github_token = userdata.get('GITHUB_TOKEN')
courtlistener_key = userdata.get('COURTLISTENER_API_KEY')

if not github_token:
    raise RuntimeError('Missing Colab Secret: GITHUB_TOKEN')
if not courtlistener_key:
    raise RuntimeError('Missing Colab Secret: COURTLISTENER_API_KEY')

# Authenticate git using an HTTP header so the token is not embedded in a URL or printed.
basic = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_auth = f'Authorization: Basic {basic}'
git_cmd = ['git', '-c', f'http.extraHeader={git_auth}']

if ROOT.exists():
    subprocess.run(git_cmd + ['-C', str(ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(git_cmd + ['-C', str(ROOT), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(git_cmd + ['clone', f'https://github.com/{REPO}.git', str(ROOT)], check=True)

os.environ['COURTLISTENER_API_KEY'] = courtlistener_key
print('Repository synced and CourtListener credential loaded securely.')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/i130-mandamus-audit/requirements.txt'], check=True)
subprocess.run([sys.executable, '/content/i130-mandamus-audit/pacer_audit.py', '--output', '/content/i130-mandamus-audit/output'], check=True)

In [ ]:
from pathlib import Path
from google.colab import files

out = Path('/content/i130-mandamus-audit/output')
print('Generated files:')
for p in sorted(out.glob('*')):
    print(' -', p.name)

# Download the main workbook to the phone/device.
files.download(str(out / 'case_audit.xlsx'))